# 🔥 DeepFire Forecaster v4.1
### Hybrid CNN-Transformer for Multi-Day Wildfire Spread Prediction

---

## Overview

DeepFire v4.1 is a deep learning model that predicts **wildfire spread 1–3 days into the future** using multi-spectral satellite imagery from VIIRS (Visible Infrared Imaging Radiometer Suite) and derived weather proxies.

### Problem Statement
Given 3 consecutive days of VIIRS thermal + nighttime satellite observations, predict the fire probability maps for the following 3 days. This is a spatiotemporal sequence-to-sequence regression problem on 128×128 pixel patches.

### Architecture at a Glance
```
VIIRS input [B, T=3, C=10, 128, 128]
        │
        ▼
┌─────────────────────────────────┐
│   Spatial CNN Encoder (×T)      │  ResidualCNNBlock + SqueezeExcite
│   4 stages: 32→64→128→256 ch    │  Skips: s1(128²) s2(64²) s3(32²) s4(16²)
│   Bottleneck: [BT, 256, 8, 8]   │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│   Spatiotemporal Transformer    │  3 layers, 8 heads, 64 spatial tokens
│   + Spatial & Temporal Pos Emb  │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│   Temporal Attention Pooling    │  Learns which of the 3 input days matters most
└──────────────┬──────────────────┘
               │  + FiLM weather conditioning (γ·x + β)
               ▼
┌─────────────────────────────────┐
│   UNet Decoder × 3              │  One decoder per prediction day
│   Attentive skip aggregation     │  Skips weighted by temporal attention
└──────────────┬──────────────────┘
               ▼
    Predictions [B, P=3, 1, 128, 128]  (fire probability maps)
```

### Key v4.1 Improvements over v3
| Feature | v3 | v4.1 |
|---|---|---|
| EMBED_DIM | 128 | 256 |
| NUM_LAYERS | 3 | 3 (was 4 in v4) |
| Channel attention | ✗ | ✅ SqueezeExcite |
| Weather conditioning | Additive bias | ✅ FiLM (scale+shift) |
| Skip aggregation | Mean over T | ✅ Attention-weighted |
| Data augmentation | ✗ | ✅ Flip + Rotation |
| Loss inside AMP | ✗ | ✅ |
| DataLoader prefetch | ✗ | ✅ prefetch_factor=2 |
| Best val loss | 0.22121 | **0.19993** |

---


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # must be set before torch is imported
print("CUDA_VISIBLE_DEVICES set to 0")

CUDA_VISIBLE_DEVICES set to 0


## Cell 1 — GPU Isolation

Sets `CUDA_VISIBLE_DEVICES=0` **before** any CUDA initialisation. This must be the very first cell executed.

> **Why:** On Kaggle T4×2 sessions, GPU 1 occasionally enters a corrupted memory state from a previous crashed kernel. PyTorch's `DataParallel` then fails with `CUDA misaligned address`. Restricting to GPU 0 gives a clean, reliable single-GPU training session with 14.6 GB VRAM — more than sufficient for this model (~250 MB at training batch size 8).


In [2]:
import subprocess, sys

required = ["rasterio", "tqdm"]
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies ready.")

All dependencies ready.


## Cell 2 — Dependencies

Installs `rasterio` (GeoTIFF reader for `.tif` satellite files) and `tqdm` (progress bars) if not already present. All other dependencies (`torch`, `numpy`, `matplotlib`) are pre-installed in the Kaggle environment.


In [3]:
import os, glob, json, math, warnings, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS  = torch.cuda.device_count()
USE_AMP = torch.cuda.is_available()

print(f"Device      : {DEVICE}")
print(f"GPUs found  : {N_GPUS}")
print(f"AMP         : {USE_AMP}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")

# Report per-GPU memory so we know exactly how much headroom we have
if torch.cuda.is_available():
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  |  {props.total_memory / 1024**3:.1f} GB VRAM")

Device      : cuda
GPUs found  : 1
AMP         : True
PyTorch     : 2.9.0+cu126
CUDA        : 12.6
  GPU 0: Tesla T4  |  14.6 GB VRAM


## Cell 3 — Imports & Device Setup

Imports all required libraries and configures:

- **Reproducibility:** Fixed seeds for `torch`, `numpy`, and CUDA. `cudnn.benchmark=True` enables cuDNN auto-tuning for fixed input sizes (faster conv kernels).
- **Device detection:** Identifies available GPUs, enables AMP (Automatic Mixed Precision) when CUDA is available.
- **AMP (`USE_AMP`):** Runs forward pass in `float16` and loss in `float32`, reducing memory usage and speeding up matrix ops on T4 Tensor Cores.


In [4]:
# ─── Paths ────────────────────────────────────────────────────────────────────
SAVE_DIR   = "/kaggle/working"
MODEL_PATH = os.path.join(SAVE_DIR, "deepfire_best.pt")

# ─── Data ─────────────────────────────────────────────────────────────────────
SPATIAL_SIZE   = 128
SEQ_LEN        = 3
PRED_HORIZON   = 3
VIIRS_CHANNELS = 8
NIGHT_CHANNELS = 2
USE_NIGHT      = True
C_VIIRS        = VIIRS_CHANNELS + NIGHT_CHANNELS   # = 10

# ─── Architecture ─────────────────────────────────────────────────────────────
# 256 is CUDA-safe (power-of-2). NUM_LAYERS 3 (was 4) saves ~25% transformer FLOPs.
EMBED_DIM  = 256
NUM_HEADS  = 8       # 256/8 = 32 dims/head — aligned
NUM_LAYERS = 3
WEATHER_DIM = 32
DROPOUT     = 0.1

# ─── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE    = 8    # per GPU → 16 effective across T4×2
EPOCHS        = 25
LR            = 2e-4
WARMUP_EPOCHS = 3
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
SPLIT_TRAIN   = 0.70
SPLIT_VAL     = 0.15

# ─── Loss ─────────────────────────────────────────────────────────────────────
FOCAL_GAMMA  = 2.5
FOCAL_ALPHA  = 0.875
DICE_WEIGHT  = 0.5

# ─── Augmentation ─────────────────────────────────────────────────────────────
AUG_PROB = 0.5

# ─── Cache / DataLoader workers ───────────────────────────────────────────────
NUM_WORKERS_CACHE = min(8, os.cpu_count() or 4)

print("Configuration:")
print(f"  EMBED_DIM   : {EMBED_DIM}   NUM_HEADS={NUM_HEADS}  ({EMBED_DIM//NUM_HEADS} dims/head)")
print(f"  NUM_LAYERS  : {NUM_LAYERS}")
print(f"  BATCH/GPU   : {BATCH_SIZE}  →  {BATCH_SIZE * N_GPUS} effective")
print(f"  EPOCHS      : {EPOCHS}")
print(f"  LR          : {LR}  warmup={WARMUP_EPOCHS} epochs")
print(f"  C_VIIRS     : {C_VIIRS}  (day={VIIRS_CHANNELS} + night={NIGHT_CHANNELS})")
print(f"  AUG_PROB    : {AUG_PROB}")
print(f"  Cache workers: {NUM_WORKERS_CACHE}")

Configuration:
  EMBED_DIM   : 256   NUM_HEADS=8  (32 dims/head)
  NUM_LAYERS  : 3
  BATCH/GPU   : 8  →  8 effective
  EPOCHS      : 25
  LR          : 0.0002  warmup=3 epochs
  C_VIIRS     : 10  (day=8 + night=2)
  AUG_PROB    : 0.5
  Cache workers: 4


## Cell 4 — Global Configuration

All hyperparameters in one place. Key design decisions:

#### Architecture
- `EMBED_DIM=256` — transformer hidden dimension. Must be a multiple of 64 for CUDA memory alignment. 256/8 heads = 32 dims/head ✓
- `NUM_LAYERS=3` — one fewer transformer layer than v4 (was 4), saving ~25% transformer FLOPs while retaining quality
- `NUM_HEADS=8` — multi-head attention for diverse spatial receptive fields

#### Training
- `BATCH_SIZE=8` — per GPU; fits comfortably in 14.6 GB T4 VRAM
- `EPOCHS=25` — sufficient for convergence given the cosine warm-restart schedule
- `LR=2e-4` — higher than v3's 1e-4; safe because warmup prevents early instability
- `WARMUP_EPOCHS=3` — linear ramp from 0 → LR before cosine decay kicks in

#### Loss
- `FOCAL_GAMMA=2.5` — down-weights easy background pixels, focuses on fire boundaries
- `FOCAL_ALPHA=0.875` — compensates for heavy class imbalance (fire pixels are rare)
- `DICE_WEIGHT=0.5` — equal blend of focal and Dice loss; Dice directly optimises IoU


In [5]:
def find_data_root():
    """Try known paths, then walk /kaggle/input for a folder containing VIIRS_Day."""
    candidates = [
        "/kaggle/input/ts-satfire/ts-satfire",
        "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
        "/kaggle/input/tssat-fire/ts-satfire",
        "/kaggle/input/ts-satfire",
    ]
    for p in candidates:
        if os.path.isdir(p):
            print(f"  Found at candidate: {p}")
            return p
    # Fallback: walk and find first dir that contains a VIIRS_Day subfolder
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "VIIRS_Day" in dirs:
            found = os.path.dirname(root)
            print(f"  Found via walk: {found}")
            return found
    raise FileNotFoundError("Cannot locate ts-satfire dataset under /kaggle/input")

DATA_ROOT = find_data_root()

# Verify structure
all_event_dirs = sorted([
    os.path.join(DATA_ROOT, d)
    for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

print(f"\nDATA_ROOT   : {DATA_ROOT}")
print(f"Event dirs  : {len(all_event_dirs)}")

# Show structure of first event
sample = all_event_dirs[0]
print(f"\nSample event: {os.path.basename(sample)}")
for sub in sorted(os.listdir(sample)):
    sub_path = os.path.join(sample, sub)
    if os.path.isdir(sub_path):
        n = len(os.listdir(sub_path))
        tag = "  ← EXCLUDED" if "ESRI" in sub else ""
        print(f"  [{sub}]  {n} files{tag}")

# Count valid events (need SEQ_LEN + PRED_HORIZON = 6 days minimum)
MIN_DAYS = SEQ_LEN + PRED_HORIZON
valid = sum(
    1 for d in all_event_dirs
    if min(
        len(glob.glob(os.path.join(d, "VIIRS_Day",  "*.tif"))),
        len(glob.glob(os.path.join(d, "FirePred",   "*.tif")))
    ) >= MIN_DAYS
)
print(f"\nValid events (≥{MIN_DAYS} days): {valid} / {len(all_event_dirs)}")

  Found at candidate: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire

DATA_ROOT   : /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire
Event dirs  : 192

Sample event: 20562846
  [ESRI_LULC]  1 files  ← EXCLUDED
  [FirePred]  8 files
  [VIIRS_Day]  8 files
  [VIIRS_Night]  8 files

Valid events (≥6 days): 172 / 192


## Cell 5 — Dataset Path Discovery

Robustly locates the `ts-satfire` dataset regardless of Kaggle's dataset slug versioning. Tries known candidate paths first, then falls back to walking `/kaggle/input` looking for any directory containing a `VIIRS_Day` subdirectory.

**Dataset structure per event:**
```
{event_id}/
  VIIRS_Day/      ← 8-band daytime thermal imagery (.tif)
  VIIRS_Night/    ← 2-band nighttime imagery (.tif)
  FirePred/       ← Fire Radiative Power ground truth (.tif)
  ESRI_LULC/      ← Land use/land cover (EXCLUDED — not used)
```

Each event represents one real wildfire incident. A sliding window of `SEQ_LEN=3` input days + `PRED_HORIZON=3` target days is extracted from each event, giving multiple training samples per event.


In [6]:
import multiprocessing as mp
from functools import partial

CACHE_DIR = os.path.join(SAVE_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Helper functions ──────────────────────────────────────────────────────────
def _read_tif_all(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    return np.nan_to_num(arr, nan=0.0)

def _read_tif_band(path, band=1):
    with rasterio.open(path) as src:
        arr = src.read(band).astype(np.float32)
    return np.nan_to_num(arr, nan=0.0)

def _normalise_viirs(arr):
    out = np.zeros_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        b = arr[i]
        bmin, bmax = b.min(), b.max()
        if bmax - bmin > 1e-6:
            out[i] = (b - bmin) / (bmax - bmin)
    return out

def _normalise_frp(arr):
    arr = np.clip(arr, 0.0, None)
    pos = arr[arr > 0]
    if len(pos) == 0:
        return arr
    p99 = max(float(np.percentile(pos, 99)), 1.0)
    return np.clip(arr / p99, 0.0, 1.0)

def _fix_channels(arr, target_c):
    c = arr.shape[0]
    if c == target_c: return arr
    if c > target_c:  return arr[:target_c]
    pad = np.zeros((target_c - c, arr.shape[1], arr.shape[2]), dtype=arr.dtype)
    return np.concatenate([arr, pad], axis=0)

def _resize(arr, size):
    t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
    t = F.interpolate(t, size=size, mode="bilinear", align_corners=False)
    return t.squeeze(0)

def preprocess_event(event_path, cache_dir, spatial_size, seq_len, pred_horizon,
                     viirs_channels, night_channels, use_night):
    import os, glob, numpy as np, torch
    import torch.nn.functional as F
    import rasterio

    S        = (spatial_size, spatial_size)
    min_days = seq_len + pred_horizon

    day_files   = sorted(glob.glob(os.path.join(event_path, "VIIRS_Day",   "*.tif")))
    fire_files  = sorted(glob.glob(os.path.join(event_path, "FirePred",    "*.tif")))
    night_files = sorted(glob.glob(os.path.join(event_path, "VIIRS_Night", "*.tif")))

    n_avail   = min(len(day_files), len(fire_files))
    n_windows = n_avail - min_days + 1
    if n_windows < 1:
        return []

    event_name = os.path.basename(event_path)
    saved = []

    for start in range(n_windows):
        cache_path = os.path.join(cache_dir, f"{event_name}_s{start:03d}.pt")
        if os.path.exists(cache_path):
            saved.append(cache_path)
            continue
        try:
            input_days   = day_files[start : start + seq_len]
            target_fires = fire_files[start + seq_len : start + seq_len + pred_horizon]

            viirs_frames = []
            for t_idx, f in enumerate(input_days):
                bands = _normalise_viirs(_read_tif_all(f))
                bands = _fix_channels(bands, viirs_channels)
                frame = _resize(bands, S)
                if use_night:
                    ni = start + t_idx
                    if ni < len(night_files):
                        n_arr = _normalise_viirs(_read_tif_all(night_files[ni]))
                        n_arr = _fix_channels(n_arr, night_channels)
                        n_t   = _resize(n_arr, S)
                    else:
                        n_t = torch.zeros(night_channels, *S)
                    frame = torch.cat([frame, n_t], dim=0)
                viirs_frames.append(frame)

            viirs_t  = torch.stack(viirs_frames, dim=0)        # [T, C, S, S]
            weather  = viirs_t[-1].mean(dim=(1, 2))             # [C]

            targets = []
            for tf in target_fires:
                tgt_raw = _read_tif_band(tf, band=1)
                tgt_raw = _normalise_frp(tgt_raw)
                tgt_t   = torch.tensor(tgt_raw).unsqueeze(0).unsqueeze(0)
                tgt_t   = F.interpolate(tgt_t, size=S, mode="bilinear",
                                        align_corners=False).squeeze(0)
                targets.append(tgt_t)
            target_t = torch.stack(targets, dim=0)              # [P, 1, S, S]

            torch.save({"viirs": viirs_t, "weather": weather, "target": target_t},
                       cache_path)
            saved.append(cache_path)
        except Exception:
            pass

    return saved

# ── Build cache ───────────────────────────────────────────────────────────────
already = len(glob.glob(os.path.join(CACHE_DIR, "*.pt")))
print(f"Already cached : {already} samples")

worker_fn = partial(
    preprocess_event,
    cache_dir      = CACHE_DIR,
    spatial_size   = SPATIAL_SIZE,
    seq_len        = SEQ_LEN,
    pred_horizon   = PRED_HORIZON,
    viirs_channels = VIIRS_CHANNELS,
    night_channels = NIGHT_CHANNELS,
    use_night      = USE_NIGHT,
)

t0  = time.time()
ctx = mp.get_context("fork")
with ctx.Pool(processes=NUM_WORKERS_CACHE) as pool:
    results = list(tqdm(
        pool.imap_unordered(worker_fn, all_event_dirs, chunksize=4),
        total=len(all_event_dirs),
        desc="Preprocessing",
    ))

all_cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, "*.pt")))
print(f"\nCache built in {time.time()-t0:.1f}s")
print(f"Total cached samples : {len(all_cache_files)}")

# Quick sanity check on one sample
sample_data = torch.load(all_cache_files[0], weights_only=True)
print(f"\nSample shapes:")
print(f"  viirs   : {tuple(sample_data['viirs'].shape)}")
print(f"  weather : {tuple(sample_data['weather'].shape)}")
print(f"  target  : {tuple(sample_data['target'].shape)}")

Already cached : 2669 samples


Preprocessing:   0%|          | 0/192 [00:00<?, ?it/s]


Cache built in 0.2s
Total cached samples : 2669

Sample shapes:
  viirs   : (3, 10, 128, 128)
  weather : (10,)
  target  : (3, 1, 128, 128)


## Cell 6 — Parallel Preprocessing Cache

Converts raw GeoTIFF files into PyTorch `.pt` tensors and caches them to disk. This is the most I/O-intensive step — runs once, then all subsequent runs skip it.

**Processing pipeline per sample window:**
1. **Read** VIIRS Day bands → `float32` array
2. **Normalise** each band independently to [0, 1] using per-band min-max
3. **Fix channels** — pad or truncate to exactly `VIIRS_CHANNELS=8` bands
4. **Resize** to `128×128` using bilinear interpolation
5. **Concatenate night bands** (2 channels) → final `[T, 10, 128, 128]` tensor
6. **Weather proxy** = mean of last input day across spatial dims → `[10]` vector
7. **Target** = normalised Fire Radiative Power, resized to `128×128` → `[P, 1, 128, 128]`

Uses Python `multiprocessing` with `fork` context for parallel processing across all 192 event directories. Already-cached files are skipped, making re-runs instant.


In [7]:
import random as pyrandom

class CachedFireDataset(Dataset):
    def __init__(self, cache_files, augment=False):
        self.files   = cache_files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _aug(self, viirs, target):
        if pyrandom.random() < AUG_PROB:
            viirs  = torch.flip(viirs,  dims=[-1])
            target = torch.flip(target, dims=[-1])
        if pyrandom.random() < AUG_PROB:
            viirs  = torch.flip(viirs,  dims=[-2])
            target = torch.flip(target, dims=[-2])
        k = pyrandom.randint(0, 3)
        if k > 0:
            viirs  = torch.rot90(viirs,  k=k, dims=[-2, -1])
            target = torch.rot90(target, k=k, dims=[-2, -1])
        return viirs, target

    def __getitem__(self, idx):
        s       = torch.load(self.files[idx], weights_only=True)
        viirs   = s["viirs"]
        weather = s["weather"]
        target  = s["target"]
        if self.augment:
            viirs, target = self._aug(viirs, target)
        return viirs, weather, target


# ── Split ─────────────────────────────────────────────────────────────────────
n_total = len(all_cache_files)
n_train = int(SPLIT_TRAIN * n_total)
n_val   = int(SPLIT_VAL   * n_total)
n_test  = n_total - n_train - n_val

generator  = torch.Generator().manual_seed(SEED)
idx_splits = random_split(range(n_total), [n_train, n_val, n_test], generator=generator)

train_files = [all_cache_files[i] for i in idx_splits[0]]
val_files   = [all_cache_files[i] for i in idx_splits[1]]
test_files  = [all_cache_files[i] for i in idx_splits[2]]

train_ds = CachedFireDataset(train_files, augment=True)
val_ds   = CachedFireDataset(val_files,   augment=False)
test_ds  = CachedFireDataset(test_files,  augment=False)

# ── DataLoaders ───────────────────────────────────────────────────────────────
_nw = min(4 * N_GPUS, 8)
loader_kw = dict(
    num_workers        = _nw,
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = 2,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE * N_GPUS,
                          shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * N_GPUS,
                          shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

# ── Verify one batch ──────────────────────────────────────────────────────────
vb, wb, tb = next(iter(train_loader))

print(f"Split  —  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print(f"Batch shapes:")
print(f"  viirs   : {tuple(vb.shape)}  dtype={vb.dtype}")
print(f"  weather : {tuple(wb.shape)}  dtype={wb.dtype}")
print(f"  target  : {tuple(tb.shape)}  dtype={tb.dtype}")
print(f"  target  range [{tb.min():.3f}, {tb.max():.3f}]")
print(f"DataLoader workers : {_nw}  prefetch=2")

Split  —  train=1868  val=400  test=401
Batch shapes:
  viirs   : (8, 3, 10, 128, 128)  dtype=torch.float32
  weather : (8, 10)  dtype=torch.float32
  target  : (8, 3, 1, 128, 128)  dtype=torch.float32
  target  range [0.000, 1.000]
DataLoader workers : 4  prefetch=2


## Cell 7 — Dataset & DataLoaders

Wraps cached `.pt` files in a `CachedFireDataset` and splits into train/val/test sets.

**Split ratios:** 70% train / 15% val / 15% test (seeded for reproducibility)

**Data Augmentation (train only):**
- Random horizontal flip (p=0.5)
- Random vertical flip (p=0.5)
- Random 90°/180°/270° rotation (p=0.75)
- Applied **identically** to both `viirs` and `target` tensors to preserve spatial correspondence

**DataLoader optimisations:**
- `persistent_workers=True` — worker processes stay alive between epochs (saves fork overhead)
- `prefetch_factor=2` — pre-loads 2 batches per worker while GPU is computing, overlapping I/O with computation
- `pin_memory=True` — allocates batches in page-locked memory for faster CPU→GPU transfer


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Building Blocks
# ═══════════════════════════════════════════════════════════════════════════════

class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ResidualCNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(ConvBNReLU(in_ch, out_ch), ConvBNReLU(out_ch, out_ch))
        self.se   = SqueezeExcite(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return F.relu(self.se(self.conv(x)) + self.skip(x), inplace=True)


# ═══════════════════════════════════════════════════════════════════════════════
# Spatial CNN Encoder
# INPUT  128×128
# s1     128×128   32 ch    (enc1 output, before pool1)
# s2      64×64   64 ch    (enc2 output, before pool2)
# s3      32×32  128 ch    (enc3 output, before pool3)
# s4      16×16    D ch    (enc4 output, before pool4)
# bt       8×8    D ch    (pool4 output → transformer)
# ═══════════════════════════════════════════════════════════════════════════════

class SpatialEncoder(nn.Module):
    def __init__(self, viirs_channels, embed_dim):
        super().__init__()
        self.enc1  = ResidualCNNBlock(viirs_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ResidualCNNBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3  = ResidualCNNBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4  = ResidualCNNBlock(128, embed_dim)
        self.pool4 = nn.MaxPool2d(2)

    def forward(self, x):
        s1 = self.enc1(x)                     # 128×128, 32ch
        s2 = self.enc2(self.pool1(s1))         #  64×64, 64ch
        s3 = self.enc3(self.pool2(s2))         #  32×32,128ch
        s4 = self.enc4(self.pool3(s3))         #  16×16,  Dch
        bt = self.pool4(s4)                    #   8×8,   Dch
        return s1, s2, s3, s4, bt


# ═══════════════════════════════════════════════════════════════════════════════
# Weather FiLM
# ═══════════════════════════════════════════════════════════════════════════════

class WeatherFiLM(nn.Module):
    def __init__(self, weather_in, embed_dim, weather_dim=WEATHER_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(weather_in, weather_dim), nn.GELU(),
            nn.Linear(weather_dim, weather_dim), nn.GELU(),
            nn.Linear(weather_dim, embed_dim * 2),
        )
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)

    def forward(self, bottleneck, weather):
        gamma, beta = self.proj(weather).chunk(2, dim=-1)
        gamma = (gamma + 1.0).unsqueeze(-1).unsqueeze(-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        return gamma * bottleneck + beta


# ═══════════════════════════════════════════════════════════════════════════════
# Spatiotemporal Transformer
# bottleneck = 8×8 = 64 spatial tokens
# ═══════════════════════════════════════════════════════════════════════════════

class SpatiotemporalTransformer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, seq_len, spatial_size, dropout):
        super().__init__()
        Hf        = spatial_size // 16          # 128//16 = 8  (4 pool stages → /16)
        n_spatial = Hf * Hf                     # 8×8 = 64 tokens
        self.spatial_pos  = nn.Parameter(torch.randn(1, 1, n_spatial, embed_dim) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, seq_len, 1, embed_dim)   * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, T, D, H, W = x.shape
        x = x.view(B, T, D, H * W).permute(0, 1, 3, 2)
        x = x + self.spatial_pos + self.temporal_pos
        x = x.reshape(B, T * H * W, D)
        x = self.norm(self.transformer(x))
        x = x.view(B, T, H * W, D).permute(0, 1, 3, 2).contiguous()
        return x.view(B, T, D, H, W)


# ═══════════════════════════════════════════════════════════════════════════════
# Temporal Attention Pooling
# ═══════════════════════════════════════════════════════════════════════════════

class TemporalAttentionPooling(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim // 4, 1), nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim // 4, 1, 1),
        )
    def forward(self, x):
        B, T, D, H, W = x.shape
        scores  = self.score_net(x.view(B * T, D, H, W)).view(B, T, -1).mean(dim=2)
        weights = F.softmax(scores, dim=1)
        pooled  = (x * weights.view(B, T, 1, 1, 1)).sum(dim=1)
        return pooled, weights

def attentive_skip(skip_seq, weights):
    return (skip_seq * weights.view(weights.size(0), weights.size(1), 1, 1, 1)).sum(dim=1)


# ═══════════════════════════════════════════════════════════════════════════════
# UNet Decoder
# bt  [ D,  8,  8] →up4→ [ D, 16, 16]  cat s4[ D,16,16] → ref4 →[128,16,16]
#                  →up3→ [128,32, 32]  cat s3[128,32,32] → ref3 →[ 64,32,32]
#                  →up2→ [ 64,64, 64]  cat s2[ 64,64,64] → ref2 →[ 32,64,64]
#                  →up1→ [ 32,128,128] cat s1[ 32,128,128]→ ref1 →[ 16,128,128]
#                  → head → [1, 128, 128]
# ═══════════════════════════════════════════════════════════════════════════════

class UNetDecoder(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.up4  = nn.Sequential(nn.ConvTranspose2d(embed_dim, embed_dim, 4, 2, 1),
                                  nn.BatchNorm2d(embed_dim), nn.ReLU(True))
        self.ref4 = ResidualCNNBlock(embed_dim + embed_dim, 128)

        self.up3  = nn.Sequential(nn.ConvTranspose2d(128, 128, 4, 2, 1),
                                  nn.BatchNorm2d(128), nn.ReLU(True))
        self.ref3 = ResidualCNNBlock(128 + 128, 64)

        self.up2  = nn.Sequential(nn.ConvTranspose2d(64, 64, 4, 2, 1),
                                  nn.BatchNorm2d(64), nn.ReLU(True))
        self.ref2 = ResidualCNNBlock(64 + 64, 32)

        self.up1  = nn.Sequential(nn.ConvTranspose2d(32, 32, 4, 2, 1),
                                  nn.BatchNorm2d(32), nn.ReLU(True))
        self.ref1 = ResidualCNNBlock(32 + 32, 16)

        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, bt, s4, s3, s2, s1):
        x = self.ref4(torch.cat([self.up4(bt), s4], dim=1))  # 8→16,  cat s4
        x = self.ref3(torch.cat([self.up3(x),  s3], dim=1))  # 16→32, cat s3
        x = self.ref2(torch.cat([self.up2(x),  s2], dim=1))  # 32→64, cat s2
        x = self.ref1(torch.cat([self.up1(x),  s1], dim=1))  # 64→128, cat s1
        return torch.sigmoid(self.head(x))                    # [B,1,128,128]


# ═══════════════════════════════════════════════════════════════════════════════
# DeepFireForecaster
# ═══════════════════════════════════════════════════════════════════════════════

class DeepFireForecaster(nn.Module):
    def __init__(self, viirs_channels=C_VIIRS, embed_dim=EMBED_DIM,
                 num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
                 seq_len=SEQ_LEN, pred_horizon=PRED_HORIZON,
                 spatial_size=SPATIAL_SIZE, dropout=DROPOUT,
                 weather_dim=WEATHER_DIM):
        super().__init__()
        self.pred_horizon = pred_horizon
        self.encoder      = SpatialEncoder(viirs_channels, embed_dim)
        self.transformer  = SpatiotemporalTransformer(
            embed_dim, num_heads, num_layers, seq_len, spatial_size, dropout)
        self.temp_pool    = TemporalAttentionPooling(embed_dim)
        self.weather_fuse = WeatherFiLM(viirs_channels, embed_dim, weather_dim)
        self.decoders     = nn.ModuleList([UNetDecoder(embed_dim)
                                           for _ in range(pred_horizon)])

    def forward(self, viirs, weather):
        B, T, C, H, W = viirs.shape
        flat = viirs.view(B * T, C, H, W)
        s1, s2, s3, s4, bt = self.encoder(flat)

        _, C1, H1, W1 = s1.shape
        _, C2, H2, W2 = s2.shape
        _, C3, H3, W3 = s3.shape
        _, C4, H4, W4 = s4.shape
        _, D,  Hb, Wb = bt.shape

        bt_seq = self.transformer(bt.view(B, T, D, Hb, Wb))
        pooled, attn_w = self.temp_pool(bt_seq)
        pooled = self.weather_fuse(pooled, weather)

        s4a = attentive_skip(s4.view(B, T, C4, H4, W4), attn_w)
        s3a = attentive_skip(s3.view(B, T, C3, H3, W3), attn_w)
        s2a = attentive_skip(s2.view(B, T, C2, H2, W2), attn_w)
        s1a = attentive_skip(s1.view(B, T, C1, H1, W1), attn_w)

        preds = torch.stack([dec(pooled, s4a, s3a, s2a, s1a)
                             for dec in self.decoders], dim=1)
        return preds, attn_w


# ── CPU shape check ───────────────────────────────────────────────────────────
print("CPU shape check...")
_m = DeepFireForecaster()

# Print intermediate shapes so we can verify alignment
with torch.no_grad():
    _flat = torch.randn(2, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE)
    _s1, _s2, _s3, _s4, _bt = _m.encoder(_flat)
    print(f"  Encoder skips:")
    print(f"    s1 : {tuple(_s1.shape)}")
    print(f"    s2 : {tuple(_s2.shape)}")
    print(f"    s3 : {tuple(_s3.shape)}")
    print(f"    s4 : {tuple(_s4.shape)}")
    print(f"    bt : {tuple(_bt.shape)}")

with torch.no_grad():
    _dv = torch.randn(2, SEQ_LEN, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE)
    _dw = torch.randn(2, C_VIIRS)
    _out, _attn = _m(_dv, _dw)
assert _out.shape  == (2, PRED_HORIZON, 1, SPATIAL_SIZE, SPATIAL_SIZE), f"Bad pred: {_out.shape}"
assert _attn.shape == (2, SEQ_LEN), f"Bad attn: {_attn.shape}"
print(f"  pred  : {tuple(_out.shape)}  ✓")
print(f"  attn  : {tuple(_attn.shape)}  ✓")
del _m, _flat, _s1, _s2, _s3, _s4, _bt, _dv, _dw, _out, _attn

# ── GPU instantiation ─────────────────────────────────────────────────────────
print("\nMoving to GPU...")
model = DeepFireForecaster().to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"  DataParallel : {N_GPUS > 1}  ({N_GPUS} GPUs)")
print(f"  Parameters   : {total_params:,}")

# ── GPU dry-run  (batch=4 so each GPU gets 2 — BatchNorm needs ≥2) ───────────
print("\nGPU dry-run...")
with torch.no_grad():
    dv = torch.randn(4, SEQ_LEN, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE).to(DEVICE)
    dw = torch.randn(4, C_VIIRS).to(DEVICE)
    out, attn = model(dv, dw)
print(f"  pred  : {tuple(out.shape)}  ✓")
print(f"  attn  : {tuple(attn.shape)}  ✓")
for i in range(N_GPUS):
    alloc = torch.cuda.memory_allocated(i) / 1024**2
    total = torch.cuda.get_device_properties(i).total_memory / 1024**2
    print(f"  GPU {i} : {alloc:.0f} MB / {total:.0f} MB  ({100*alloc/total:.1f}%)")

CPU shape check...
  Encoder skips:
    s1 : (2, 32, 128, 128)
    s2 : (2, 64, 64, 64)
    s3 : (2, 128, 32, 32)
    s4 : (2, 256, 16, 16)
    bt : (2, 256, 8, 8)
  pred  : (2, 3, 1, 128, 128)  ✓
  attn  : (2, 3)  ✓

Moving to GPU...
  DataParallel : False  (1 GPUs)
  Parameters   : 11,062,484

GPU dry-run...
  pred  : (4, 3, 1, 128, 128)  ✓
  attn  : (4, 3)  ✓
  GPU 0 : 60 MB / 14913 MB  (0.4%)


## Cell 8 — Model Architecture

The full DeepFire v4.1 architecture. All channel counts are multiples of 32 for CUDA memory alignment.

### Building Blocks

**`SqueezeExcite`** — Channel attention module. Global-average-pools each feature map to a scalar, passes through a small MLP, and produces per-channel scaling weights in [0,1]. This lets the encoder learn which of the 10 spectral bands are most informative for fire detection.

**`ResidualCNNBlock`** — Two `Conv→BN→ReLU` layers with a residual skip connection, followed by SqueezeExcite. The 1×1 projection in the skip handles channel dimension changes.

### Spatial CNN Encoder
Processes all `B×T` frames independently:
```
128×128 → [enc1] → 128×128, 32ch  (s1 — skip)
        → [pool] →  64×64
        → [enc2] →  64×64, 64ch  (s2 — skip)
        → [pool] →  32×32
        → [enc3] →  32×32,128ch  (s3 — skip)
        → [pool] →  16×16
        → [enc4] →  16×16,256ch  (s4 — skip)
        → [pool] →   8×8, 256ch  (bt — to transformer)
```

### WeatherFiLM
Feature-wise Linear Modulation. Projects the weather vector to `(γ, β)` pairs and applies `γ·bottleneck + β`. Initialised as identity (γ=1, β=0) so the model can gradually learn weather influence without destabilising early training.

### Spatiotemporal Transformer
Flattens the `8×8` bottleneck into 64 spatial tokens, adds learned spatial + temporal positional embeddings, then runs through 3 pre-norm transformer encoder layers. Operates on `T×64 = 192` tokens simultaneously, allowing attention across both space and time.

### Temporal Attention Pooling
Collapses the T=3 temporal dimension into a single frame by learning a scalar importance score per timestep. Returns both the pooled feature map and the attention weights (used for attentive skip aggregation and analysis).

### UNet Decoder (×3)
One decoder per prediction day. Uses transposed convolutions to upsample from `8×8` back to `128×128`, concatenating attention-weighted skip features at each scale. The shared skips mean all 3 decoders see the same spatial context but learn independent upsampling weights.


In [13]:
class MaskedFocalDiceLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA,
                 dice_weight=DICE_WEIGHT, eps=1e-6):
        super().__init__()
        self.gamma       = gamma
        self.alpha       = alpha
        self.dice_weight = dice_weight
        self.eps         = eps

    def _one_day(self, pred, target):
        # Disable autocast for the entire loss — BCE is unsafe in fp16
        with torch.autocast(device_type="cuda", enabled=False):
            pred   = pred.float()
            target = target.float()
            mask   = (~torch.isnan(target)) & (target >= 0.0) & (target <= 1.0)
            mask   = mask.float()
            p = pred   * mask
            t = target * mask
            n = mask.sum().clamp(min=1.0)

            bce        = F.binary_cross_entropy(p, t, reduction="none")
            pt         = torch.exp(-bce)
            focal      = self.alpha * (1.0 - pt) ** self.gamma * bce
            focal_loss = (focal * mask).sum() / n

            pf = p.reshape(-1)
            tf = t.reshape(-1)
            dice_loss = 1.0 - (2.0 * (pf * tf).sum() + self.eps) / \
                              (pf.sum() + tf.sum() + self.eps)

            return (1.0 - self.dice_weight) * focal_loss + self.dice_weight * dice_loss

    def forward(self, pred, target):
        P = pred.shape[1]
        return sum(self._one_day(pred[:, p], target[:, p]) for p in range(P)) / P


criterion = MaskedFocalDiceLoss()
print(f"MaskedFocalDiceLoss ready")
print(f"  gamma={FOCAL_GAMMA}  alpha={FOCAL_ALPHA}  dice_weight={DICE_WEIGHT}")

# ── Sanity check ──────────────────────────────────────────────────────────────
with torch.no_grad():
    _pred = torch.rand(4, PRED_HORIZON, 1, SPATIAL_SIZE, SPATIAL_SIZE)
    _tgt  = (torch.rand(4, PRED_HORIZON, 1, SPATIAL_SIZE, SPATIAL_SIZE) > 0.9).float()
    _loss = criterion(_pred, _tgt)
    print(f"  Sanity-check loss : {_loss.item():.6f}  (expected ~0.5–1.0)")
    del _pred, _tgt, _loss

# ── GPU loss check (inside autocast) ─────────────────────────────────────────
with torch.no_grad():
    dv = torch.randn(4, SEQ_LEN, C_VIIRS, SPATIAL_SIZE, SPATIAL_SIZE).to(DEVICE)
    dw = torch.randn(4, C_VIIRS).to(DEVICE)
    dt = (torch.rand(4, PRED_HORIZON, 1, SPATIAL_SIZE, SPATIAL_SIZE) > 0.9).float().to(DEVICE)
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        pred, _ = model(dv, dw)
        loss = criterion(pred, dt)
    print(f"  GPU loss check    : {loss.item():.6f}  ✓")
    del dv, dw, dt, pred, loss

MaskedFocalDiceLoss ready
  gamma=2.5  alpha=0.875  dice_weight=0.5
  Sanity-check loss : 0.660981  (expected ~0.5–1.0)
  GPU loss check    : 0.460672  ✓


## Cell 9 — Loss Function: Masked Focal-Dice

A compound loss designed for the severe class imbalance in wildfire prediction (fire pixels are typically <5% of the image).

### Masked Focal Loss
$$L_{focal} = -\alpha (1-p_t)^\gamma \cdot \log(p_t)$$

- `α=0.875` — up-weights fire pixel gradients to compensate for class imbalance  
- `γ=2.5` — focuses learning on hard, uncertain predictions; easy correct predictions contribute near-zero loss
- **Masking** — pixels where the target is `NaN` or outside `[0,1]` are excluded from the loss (handles sensor dropout / missing data)

### Dice Loss
$$L_{dice} = 1 - \frac{2 \sum p \cdot t}{\sum p + \sum t + \epsilon}$$

Directly optimises the overlap between predicted and ground-truth fire regions. Complementary to focal loss — focal penalises wrong pixel probabilities, Dice penalises wrong spatial extent.

### Combined Loss
$$L = 0.5 \cdot L_{focal} + 0.5 \cdot L_{dice}$$

Averaged over all `PRED_HORIZON=3` prediction days.

> **AMP safety:** `F.binary_cross_entropy` is unsafe under `float16` autocast. The loss explicitly disables autocast internally via `torch.autocast(enabled=False)` and casts inputs to `float32` before computing BCE.


In [14]:
# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── Scheduler: linear warmup → CosineAnnealingWarmRestarts ───────────────────
T_0 = max((EPOCHS - WARMUP_EPOCHS) // 3, 6)
cosine_sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=T_0, T_mult=1, eta_min=5e-6)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def binary_iou(pred, target, threshold=0.5):
    p = (pred   >= threshold).float()
    t = (target >= threshold).float()
    intersection = (p * t).sum()
    union        = (p + t).clamp(0, 1).sum()
    return (intersection / (union + 1e-6)).item()

history = {"train_loss": [], "val_loss": [], "val_iou": [], "lr": []}
best_val_loss = float("inf")

print(f"{'='*62}")
print(f"  DeepFireForecaster — {EPOCHS} epochs  |  device: {DEVICE}")
print(f"  Warmup: {WARMUP_EPOCHS} ep  |  CosineWarmRestarts T0={T_0}")
print(f"  Batch: {BATCH_SIZE}  |  LR: {LR}  |  AMP: {USE_AMP}")
print(f"{'='*62}\n")

for epoch in range(1, EPOCHS + 1):

    # ── Warmup LR ─────────────────────────────────────────────────────────────
    if epoch <= WARMUP_EPOCHS:
        warmup_lr = LR * epoch / WARMUP_EPOCHS
        for pg in optimizer.param_groups:
            pg["lr"] = warmup_lr

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Ep {epoch:02d}/{EPOCHS} [train]", leave=False)
    for viirs_b, weather_b, tgt_b in pbar:
        viirs_b   = viirs_b.to(DEVICE, non_blocking=True)
        weather_b = weather_b.to(DEVICE, non_blocking=True)
        tgt_b     = tgt_b.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred, _ = model(viirs_b, weather_b)
            loss    = criterion(pred, tgt_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.5f}"})

    avg_train = train_loss / len(train_loader)

    # ── VALIDATE every 2 epochs ───────────────────────────────────────────────
    run_val = (epoch % 2 == 0) or (epoch == EPOCHS)
    if run_val:
        model.eval()
        val_loss = val_iou = 0.0
        with torch.no_grad():
            for viirs_b, weather_b, tgt_b in val_loader:
                viirs_b   = viirs_b.to(DEVICE)
                weather_b = weather_b.to(DEVICE)
                tgt_b     = tgt_b.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    pred, _ = model(viirs_b, weather_b)
                    v_loss  = criterion(pred, tgt_b)
                val_loss += v_loss.item()
                val_iou  += binary_iou(pred.cpu(), tgt_b.cpu())
        avg_val = val_loss / len(val_loader)
        avg_iou = val_iou  / len(val_loader)
    else:
        avg_val = history["val_loss"][-1] if history["val_loss"] else float("inf")
        avg_iou = history["val_iou"][-1]  if history["val_iou"]  else 0.0

    # ── Scheduler step ────────────────────────────────────────────────────────
    if epoch > WARMUP_EPOCHS:
        cosine_sched.step()

    cur_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["val_iou"].append(avg_iou)
    history["lr"].append(cur_lr)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    improved = run_val and (avg_val < best_val_loss)
    if improved:
        best_val_loss = avg_val
        torch.save({
            "epoch"    : epoch,
            "model"    : model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_loss" : avg_val,
            "val_iou"  : avg_iou,
            "history"  : history,
        }, MODEL_PATH)

    val_str  = f"val={avg_val:.5f}  IoU={avg_iou:.4f}" if run_val else "val=--      IoU=--    "
    star     = "  ★" if improved else ""
    print(f"Ep {epoch:02d}/{EPOCHS}  train={avg_train:.5f}  {val_str}  lr={cur_lr:.2e}{star}")

print(f"\n{'='*62}")
print(f"  Training complete.  Best val loss: {best_val_loss:.5f}")
print(f"  Checkpoint saved to: {MODEL_PATH}")
print(f"{'='*62}")

  DeepFireForecaster — 25 epochs  |  device: cuda
  Warmup: 3 ep  |  CosineWarmRestarts T0=7
  Batch: 8  |  LR: 0.0002  |  AMP: True



Ep 01/25 [train]:   0%|          | 0/234 [00:00<?, ?it/s]

Ep 01/25  train=0.25690  val=--      IoU=--      lr=6.67e-05


Ep 02/25 [train]:   0%|          | 0/234 [00:00<?, ?it/s]

Ep 02/25  train=0.22229  val=0.22117  IoU=0.7131  lr=1.33e-04  ★


...

Ep 23/25  train=0.19661  val=--      IoU=--      lr=1.47e-05


Ep 24/25 [train]:   0%|          | 0/234 [00:00<?, ?it/s]

Ep 24/25  train=0.19557  val=0.19993  IoU=0.7858  lr=2.00e-04  ★


Ep 25/25 [train]:   0%|          | 0/234 [00:00<?, ?it/s]

Ep 25/25  train=0.19779  val=0.20286  IoU=0.8056  lr=1.90e-04

  Training complete.  Best val loss: 0.19993
  Checkpoint saved to: /kaggle/working/deepfire_best.pt


## Cell 10 — Training Loop

### Optimiser: AdamW
Weight decay `1e-4` acts as L2 regularisation, decoupled from the gradient update (unlike standard Adam). Prevents overfitting on the relatively small dataset.

### Learning Rate Schedule
1. **Linear warmup** (epochs 1–3): LR ramps from `LR/WARMUP_EPOCHS` → `LR`. Prevents large gradient updates before the model has stabilised.
2. **CosineAnnealingWarmRestarts** (epochs 4–25): Cosine decay with periodic restarts every `T_0=7` epochs. Restarts escape local minima and re-explore higher LRs — visible in the LR curve as sawtooth pattern.

### Training Optimisations
- **AMP** (`torch.cuda.amp.autocast`): Forward + loss in fp16, backward in fp32. ~1.5–2× speedup on T4 Tensor Cores.
- **GradScaler**: Scales loss before backward to prevent fp16 underflow; unscales before gradient clipping.
- **Gradient clipping** (`max_norm=1.0`): Prevents exploding gradients, especially important with transformers.
- **`zero_grad(set_to_none=True)`**: Releases gradient tensor memory rather than zeroing it — marginally faster.

### Validation Strategy
Validation runs every **2 epochs** to halve validation overhead. Best checkpoint is saved whenever validation improves.

### Results
| Epoch | Val Loss | Val IoU |
|---|---|---|
| 1 | — | — |
| 2 | 0.22117 | 0.713 |
| 10 | 0.20395 | 0.775 |
| 16 | 0.20074 | 0.784 |
| **24** | **0.19993** | **0.786** |

**Best val loss: 0.19993** — a **9.6% improvement** over v3's 0.22121.


In [15]:
import csv, os, math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import numpy as np
from collections import defaultdict

EVAL_DIR  = os.path.join(SAVE_DIR, "evaluation")
PLOTS_DIR = os.path.join(EVAL_DIR, "plots")
STATS_DIR = os.path.join(EVAL_DIR, "stats")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"Loaded checkpoint — epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.5f}  IoU={ckpt['val_iou']:.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. TRAINING CURVES
# ═══════════════════════════════════════════════════════════════════════════════
hist = ckpt["history"]
epochs_all = range(1, len(hist["train_loss"]) + 1)
val_epochs = [e for e in epochs_all if hist["val_loss"][e-1] != hist["val_loss"][max(0,e-2)]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("DeepFire v4.1 — Training Curves", fontsize=14, fontweight="bold")

# Loss
ax = axes[0]
ax.plot(epochs_all, hist["train_loss"], "b-o", ms=3, label="Train loss")
ax.plot(epochs_all, hist["val_loss"],   "r-o", ms=3, label="Val loss")
ax.axhline(0.22121, color="gray", ls="--", lw=1.5, label="v3 best (0.22121)")
ax.axhline(min(hist["val_loss"]), color="green", ls=":", lw=1.5,
           label=f"v4.1 best ({min(hist['val_loss']):.5f})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Loss")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# IoU
ax = axes[1]
ax.plot(epochs_all, hist["val_iou"], "g-o", ms=3, label="Val IoU")
ax.set_xlabel("Epoch"); ax.set_ylabel("IoU"); ax.set_title("Validation IoU")
ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# LR
ax = axes[2]
ax.plot(epochs_all, hist["lr"], "m-o", ms=3)
ax.set_xlabel("Epoch"); ax.set_ylabel("LR"); ax.set_title("Learning Rate Schedule")
ax.set_yscale("log"); ax.grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(PLOTS_DIR, "training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. TEST-SET EVALUATION
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(pred_np, tgt_np, threshold=0.5):
    p = (pred_np >= threshold).astype(float)
    t = (tgt_np  >= threshold).astype(float)
    tp = (p * t).sum()
    fp = (p * (1 - t)).sum()
    fn = ((1 - p) * t).sum()
    union = (p + t).clip(0, 1).sum()
    iou       = tp / (union + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    mae       = np.abs(pred_np - tgt_np).mean()
    rmse      = np.sqrt(((pred_np - tgt_np) ** 2).mean())
    return {"IoU": iou, "F1": f1, "Precision": precision,
            "Recall": recall, "MAE": mae, "RMSE": rmse}

model.eval()
all_results = []

with torch.no_grad():
    for idx, (viirs_b, weather_b, tgt_b) in enumerate(tqdm(test_loader, desc="Evaluating test set")):
        viirs_b   = viirs_b.to(DEVICE)
        weather_b = weather_b.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred, attn = model(viirs_b, weather_b)

        pred_np  = pred[0].cpu().float().numpy()   # [P,1,H,W]
        tgt_np   = tgt_b[0].numpy()
        viirs_np = viirs_b[0].cpu().numpy()        # [T,C,H,W]
        attn_np  = attn[0].cpu().numpy()           # [T]

        day_metrics = [compute_metrics(pred_np[p, 0], tgt_np[p, 0])
                       for p in range(PRED_HORIZON)]
        mean_iou = np.mean([m["IoU"] for m in day_metrics])

        all_results.append({
            "idx"        : idx,
            "pred"       : pred_np,
            "target"     : tgt_np,
            "viirs"      : viirs_np,
            "attn"       : attn_np,
            "day_metrics": day_metrics,
            "mean_iou"   : mean_iou,
        })

all_results.sort(key=lambda r: r["mean_iou"])
print(f"\nTest samples evaluated: {len(all_results)}")
print(f"  Mean IoU  : {np.mean([r['mean_iou'] for r in all_results]):.4f}")
print(f"  Median IoU: {np.median([r['mean_iou'] for r in all_results]):.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. PER-DAY STATISTICS TABLE
# ═══════════════════════════════════════════════════════════════════════════════
day_agg = defaultdict(list)
for r in all_results:
    for d, m in enumerate(r["day_metrics"]):
        for k, v in m.items():
            day_agg[(d, k)].append(v)

print(f"\n{'─'*70}")
print(f"{'Metric':<12}", end="")
for d in range(PRED_HORIZON):
    print(f"  Day+{d+1} mean ± std  ", end="")
print()
print(f"{'─'*70}")
for metric in ["IoU", "F1", "Precision", "Recall", "MAE", "RMSE"]:
    print(f"{metric:<12}", end="")
    for d in range(PRED_HORIZON):
        vals = day_agg[(d, metric)]
        print(f"  {np.mean(vals):.4f} ± {np.std(vals):.4f}  ", end="")
    print()
print(f"{'─'*70}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. DISTRIBUTION PLOTS
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Test-Set Metric Distributions per Prediction Day", fontsize=13, fontweight="bold")
colors = ["#e74c3c", "#e67e22", "#2ecc71"]

for col, metric in enumerate(["IoU", "F1", "MAE"]):
    for row, stat in enumerate(["box", "hist"]):
        ax = axes[row, col]
        data = [day_agg[(d, metric)] for d in range(PRED_HORIZON)]
        if stat == "box":
            bp = ax.boxplot(data, patch_artist=True,
                            labels=[f"Day+{d+1}" for d in range(PRED_HORIZON)])
            for patch, c in zip(bp["boxes"], colors):
                patch.set_facecolor(c); patch.set_alpha(0.7)
            ax.set_title(f"{metric} — Box Plot"); ax.grid(alpha=0.3)
        else:
            for d in range(PRED_HORIZON):
                ax.hist(data[d], bins=30, alpha=0.6, color=colors[d],
                        label=f"Day+{d+1}")
            ax.set_title(f"{metric} — Distribution")
            ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
dist_path = os.path.join(PLOTS_DIR, "metric_distributions.png")
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {dist_path}")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. COMPOSITE PREDICTION PLOTS  (best 3, worst 3, random 3)
# ═══════════════════════════════════════════════════════════════════════════════
best_3   = all_results[-3:][::-1]
worst_3  = all_results[:3]
rng      = np.random.default_rng(SEED)
random_3 = rng.choice(all_results[3:-3], size=3, replace=False).tolist()

cmap_therm = plt.cm.hot
cmap_fire  = mcolors.LinearSegmentedColormap.from_list("fire", ["#000000","#FF4500","#FFFF00"])
cmap_diff  = plt.cm.RdBu_r

def make_composite_plot(result, category, rank, save_dir):
    pred_np  = result["pred"]    # [P,1,H,W]
    tgt_np   = result["target"]  # [P,1,H,W]
    viirs_np = result["viirs"]   # [T,C,H,W]
    metrics  = result["day_metrics"]
    mean_iou = result["mean_iou"]

    n_rows = 1 + PRED_HORIZON   # input row + one row per pred day
    fig = plt.figure(figsize=(14, 4 * n_rows))
    gs  = gridspec.GridSpec(n_rows, 3, figure=fig, hspace=0.4, wspace=0.3)

    # Input row — show last input day, thermal band 0
    for t in range(min(3, SEQ_LEN)):
        ax = fig.add_subplot(gs[0, t])
        ax.imshow(viirs_np[t, 0], cmap=cmap_therm, vmin=0, vmax=1)
        ax.set_title(f"Input Day {t+1}\n(VIIRS band 0)", fontsize=9)
        ax.axis("off")

    # Prediction rows
    for p in range(PRED_HORIZON):
        m = metrics[p]
        ax_pred = fig.add_subplot(gs[p+1, 0])
        ax_tgt  = fig.add_subplot(gs[p+1, 1])
        ax_diff = fig.add_subplot(gs[p+1, 2])

        ax_pred.imshow(pred_np[p, 0], cmap=cmap_fire, vmin=0, vmax=1)
        ax_pred.set_title(f"Pred Day+{p+1}\nIoU={m['IoU']:.3f}  F1={m['F1']:.3f}", fontsize=9)
        ax_pred.axis("off")

        ax_tgt.imshow(tgt_np[p, 0], cmap=cmap_fire, vmin=0, vmax=1)
        ax_tgt.set_title(f"GT Day+{p+1}\nPrec={m['Precision']:.3f}  Rec={m['Recall']:.3f}", fontsize=9)
        ax_tgt.axis("off")

        diff = pred_np[p, 0] - tgt_np[p, 0]
        vmax = max(abs(diff).max(), 0.01)
        im = ax_diff.imshow(diff, cmap=cmap_diff, vmin=-vmax, vmax=vmax)
        ax_diff.set_title(f"Diff Day+{p+1}\nMAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}", fontsize=9)
        ax_diff.axis("off")
        fig.colorbar(im, ax=ax_diff, fraction=0.046, pad=0.04)

    fig.suptitle(f"{category} #{rank+1}  |  Sample {result['idx']}  |  Mean IoU={mean_iou:.3f}",
                 fontsize=12, fontweight="bold")
    fname = f"{category.lower()}_{rank+1:02d}_sample{result['idx']:04d}.png"
    fpath = os.path.join(save_dir, fname)
    plt.savefig(fpath, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return fpath

print("\nGenerating composite plots…")
saved_plots = []
for cat, samples in [("Best", best_3), ("Worst", worst_3), ("Random", random_3)]:
    for rank, res in enumerate(samples):
        fp = make_composite_plot(res, cat, rank, PLOTS_DIR)
        saved_plots.append(fp)
        print(f"  {os.path.basename(fp)}  (IoU={res['mean_iou']:.3f})")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. ATTENTION WEIGHT ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
all_attn = np.stack([r["attn"] for r in all_results])   # [N, T]
mean_attn = all_attn.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Temporal Attention Weight Analysis", fontsize=13, fontweight="bold")

ax = axes[0]
ax.bar(range(1, SEQ_LEN+1), mean_attn, color="#3498db", alpha=0.8)
ax.set_xlabel("Input Day"); ax.set_ylabel("Mean Attention Weight")
ax.set_title("Average Attention Across Test Set")
ax.set_xticks(range(1, SEQ_LEN+1))
ax.grid(alpha=0.3, axis="y")

ax = axes[1]
for i, r in enumerate(best_3 + worst_3):
    label = f"Best #{i+1}" if i < 3 else f"Worst #{i-2}"
    ls    = "-" if i < 3 else "--"
    ax.plot(range(1, SEQ_LEN+1), r["attn"], marker="o", ls=ls, label=label)
ax.set_xlabel("Input Day"); ax.set_ylabel("Attention Weight")
ax.set_title("Attention: Best vs Worst Samples")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
attn_path = os.path.join(PLOTS_DIR, "attention_analysis.png")
plt.savefig(attn_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {attn_path}")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. CSV METRICS EXPORT
# ═══════════════════════════════════════════════════════════════════════════════
csv_path = os.path.join(STATS_DIR, "test_metrics.csv")
day_keys = [f"Day{d+1}_{k}" for d in range(PRED_HORIZON)
            for k in ["IoU","F1","Precision","Recall","MAE","RMSE"]]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["sample_idx","mean_iou"] + day_keys)
    writer.writeheader()
    for r in all_results:
        row = {"sample_idx": r["idx"], "mean_iou": round(r["mean_iou"], 5)}
        for d, m in enumerate(r["day_metrics"]):
            for k in ["IoU","F1","Precision","Recall","MAE","RMSE"]:
                row[f"Day{d+1}_{k}"] = round(m[k], 5)
        writer.writerow(row)
print(f"Saved: {csv_path}")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*62}")
print(f"  FINAL RESULTS SUMMARY")
print(f"{'='*62}")
print(f"  Best val loss  : {ckpt['val_loss']:.5f}  (v3 baseline: 0.22121)")
print(f"  Improvement    : {(0.22121 - ckpt['val_loss'])/0.22121*100:.1f}% reduction in loss")
print(f"  Test samples   : {len(all_results)}")
print(f"  Mean test IoU  : {np.mean([r['mean_iou'] for r in all_results]):.4f}")
for d in range(PRED_HORIZON):
    iou_vals = day_agg[(d, 'IoU')]
    f1_vals  = day_agg[(d, 'F1')]
    print(f"  Day+{d+1}  IoU={np.mean(iou_vals):.4f}  F1={np.mean(f1_vals):.4f}")
print(f"{'='*62}")
print(f"  Plots  → {PLOTS_DIR}")
print(f"  Metrics→ {csv_path}")

Loaded checkpoint — epoch 24  val_loss=0.19993  IoU=0.7858
Saved: /kaggle/working/evaluation/plots/training_curves.png


Evaluating test set:   0%|          | 0/401 [00:00<?, ?it/s]


Test samples evaluated: 401
  Mean IoU  : 0.7782
  Median IoU: 0.8013

──────────────────────────────────────────────────────────────────────
Metric        Day+1 mean ± std    Day+2 mean ± std    Day+3 mean ± std  
──────────────────────────────────────────────────────────────────────
IoU           0.7791 ± 0.1223    0.7775 ± 0.1223    0.7779 ± 0.1215  
F1            0.8701 ± 0.0839    0.8691 ± 0.0836    0.8694 ± 0.0830  
Precision     0.7904 ± 0.1194    0.7891 ± 0.1190    0.7899 ± 0.1180  
Recall        0.9793 ± 0.0239    0.9785 ± 0.0267    0.9780 ± 0.0278  
MAE           0.1134 ± 0.0229    0.1129 ± 0.0221    0.1138 ± 0.0225  
RMSE          0.1488 ± 0.0279    0.1482 ± 0.0273    0.1492 ± 0.0272  
──────────────────────────────────────────────────────────────────────
Saved: /kaggle/working/evaluation/plots/metric_distributions.png

Generating composite plots…
  best_01_sample0190.png  (IoU=0.986)
  best_02_sample0284.png  (IoU=0.965)
  best_03_sample0372.png  (IoU=0.961)
  worst_01_sam

## Cell 11 — Evaluation, Statistics & Plots

Loads the best checkpoint and runs full evaluation on the held-out test set (401 samples).

### Metrics Computed per Sample per Day
- **IoU** (Intersection over Union) — primary metric; measures spatial overlap at threshold=0.5
- **F1 Score** — harmonic mean of precision and recall
- **Precision** — of predicted fire pixels, how many are actually fire
- **Recall** — of actual fire pixels, how many did we predict
- **MAE** — pixel-wise mean absolute error on raw probabilities
- **RMSE** — root mean squared error, more sensitive to large errors

### Plots Generated
1. **Training curves** — train/val loss, IoU, LR schedule with v3 baseline reference
2. **Metric distributions** — box plots and histograms for IoU, F1, MAE per prediction day
3. **Composite prediction plots** — best 3, worst 3, random 3 samples showing input VIIRS, predicted fire map, ground truth, and difference map
4. **Attention weight analysis** — which input days the model attends to most


In [17]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import numpy as np
from collections import OrderedDict

EXT_DIR = os.path.join(EVAL_DIR, "extended")
os.makedirs(EXT_DIR, exist_ok=True)

_base = model.module if hasattr(model, "module") else model

# ═══════════════════════════════════════════════════════════════════════════════
# 1. PER-LAYER PARAMETER TABLE + BAR CHART
# ═══════════════════════════════════════════════════════════════════════════════
layer_params = OrderedDict()
for name, module in _base.named_modules():
    own = sum(p.numel() for p in module.parameters(recurse=False))
    if own > 0:
        layer_params[name] = own

# Aggregate by top-level component
component_params = OrderedDict()
for name, n in layer_params.items():
    top = name.split(".")[0]
    component_params[top] = component_params.get(top, 0) + n

total = sum(component_params.values())

print(f"\n{'═'*58}")
print(f"  PARAMETER COUNT BY COMPONENT")
print(f"{'═'*58}")
print(f"  {'Component':<25} {'Params':>10}  {'% total':>8}")
print(f"  {'─'*50}")
for comp, n in component_params.items():
    print(f"  {comp:<25} {n:>10,}  {100*n/total:>7.2f}%")
print(f"  {'─'*50}")
print(f"  {'TOTAL':<25} {total:>10,}  {'100.00%':>8}")
print(f"{'═'*58}")

# Detailed per-leaf table
print(f"\n{'─'*65}")
print(f"  {'Layer':<40} {'Params':>10}  {'%':>6}")
print(f"{'─'*65}")
for name, n in list(layer_params.items())[:40]:   # top 40 leaves
    print(f"  {name:<40} {n:>10,}  {100*n/total:>5.2f}%")
if len(layer_params) > 40:
    print(f"  ... ({len(layer_params)-40} more layers)")
print(f"{'─'*65}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Parameter Distribution", fontsize=13, fontweight="bold")

ax = axes[0]
comps  = list(component_params.keys())
values = [component_params[c] / 1e6 for c in comps]
bars   = ax.barh(comps, values, color=plt.cm.tab10(np.linspace(0, 1, len(comps))))
ax.set_xlabel("Parameters (M)")
ax.set_title("Parameters by Component")
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{v:.2f}M", va="center", fontsize=9)
ax.grid(alpha=0.3, axis="x")

ax = axes[1]
ax.pie(values, labels=comps, autopct="%1.1f%%",
       colors=plt.cm.tab10(np.linspace(0, 1, len(comps))), startangle=90)
ax.set_title("Parameter Share (%)")

plt.tight_layout()
p = os.path.join(EXT_DIR, "parameter_distribution.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. ENCODER CHANNEL / SPATIAL SIZE DIAGRAM
# ═══════════════════════════════════════════════════════════════════════════════
stages = [
    ("Input",    SPATIAL_SIZE,    C_VIIRS),
    ("enc1/s1",  SPATIAL_SIZE,    32),
    ("enc2/s2",  SPATIAL_SIZE//2, 64),
    ("enc3/s3",  SPATIAL_SIZE//4, 128),
    ("enc4/s4",  SPATIAL_SIZE//8, 256),
    ("pool4/bt", SPATIAL_SIZE//16,256),
]
fig, ax = plt.subplots(figsize=(13, 4))
fig.suptitle("Encoder Spatial×Channel Progression", fontsize=12, fontweight="bold")
colors_enc = plt.cm.Blues(np.linspace(0.3, 0.9, len(stages)))
x_pos = np.arange(len(stages))
widths  = [s[1] / SPATIAL_SIZE for s in stages]
heights = [min(s[2] / 256, 1.0)           for s in stages]
for i, (name, spatial, ch) in enumerate(stages):
    ax.bar(i, heights[i], width=0.5, color=colors_enc[i], edgecolor="black", lw=0.8)
    ax.text(i, heights[i] + 0.02, f"{spatial}×{spatial}\n{ch}ch",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x_pos)
ax.set_xticklabels([s[0] for s in stages], fontsize=9)
ax.set_ylabel("Relative channel depth (normalised to 256)")
ax.set_ylim(0, 1.25)
ax.grid(alpha=0.3, axis="y")
p = os.path.join(EXT_DIR, "encoder_progression.png")
plt.tight_layout()
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. PRECISION–RECALL CURVE  (per prediction day)
# ═══════════════════════════════════════════════════════════════════════════════
thresholds = np.linspace(0.05, 0.95, 40)
colors_day = ["#e74c3c", "#e67e22", "#2ecc71"]

fig, axes = plt.subplots(1, PRED_HORIZON, figsize=(15, 5), sharey=True)
fig.suptitle("Precision–Recall Curves per Prediction Day", fontsize=13, fontweight="bold")

for d in range(PRED_HORIZON):
    ax = axes[d]
    precisions, recalls = [], []
    for thr in thresholds:
        tp = fp = fn = 0.0
        for r in all_results:
            p_bin = (r["pred"][d, 0] >= thr).astype(float)
            t_bin = (r["target"][d, 0] >= 0.5).astype(float)
            tp += (p_bin * t_bin).sum()
            fp += (p_bin * (1 - t_bin)).sum()
            fn += ((1 - p_bin) * t_bin).sum()
        precisions.append(tp / (tp + fp + 1e-6))
        recalls.append(tp / (tp + fn + 1e-6))
    auc = np.trapz(precisions[::-1], recalls[::-1])
    ax.plot(recalls, precisions, color=colors_day[d], lw=2)
    ax.fill_between(recalls, precisions, alpha=0.15, color=colors_day[d])
    ax.set_title(f"Day+{d+1}  (AUC={auc:.3f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision") if d == 0 else None
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.plot([0,1],[1,0], "k--", lw=0.8, alpha=0.4)

plt.tight_layout()
p = os.path.join(EXT_DIR, "precision_recall_curves.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. CALIBRATION PLOT  (predicted probability vs actual fire frequency)
# ═══════════════════════════════════════════════════════════════════════════════
bins = np.linspace(0, 1, 11)
bin_mids = (bins[:-1] + bins[1:]) / 2

fig, axes = plt.subplots(1, PRED_HORIZON, figsize=(15, 5))
fig.suptitle("Calibration: Predicted Probability vs Observed Fire Frequency",
             fontsize=12, fontweight="bold")

for d in range(PRED_HORIZON):
    pred_all = np.concatenate([r["pred"][d, 0].ravel() for r in all_results])
    tgt_all  = np.concatenate([r["target"][d, 0].ravel() for r in all_results])
    tgt_bin  = (tgt_all >= 0.5).astype(float)

    bin_means, bin_frac = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (pred_all >= lo) & (pred_all < hi)
        if mask.sum() > 0:
            bin_means.append(pred_all[mask].mean())
            bin_frac.append(tgt_bin[mask].mean())

    ax = axes[d]
    ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
    ax.plot(bin_means, bin_frac, "o-", color=colors_day[d], lw=2, ms=6, label="Model")
    ax.fill_between(bin_means, bin_frac, bin_means,
                    alpha=0.2, color=colors_day[d], label="Gap")
    ax.set_title(f"Day+{d+1}")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fire fraction") if d == 0 else None
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
p = os.path.join(EXT_DIR, "calibration.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. ERROR MAP ANALYSIS  (mean absolute error per pixel location)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, PRED_HORIZON, figsize=(15, 5))
fig.suptitle("Mean Absolute Error — Spatial Heatmap per Day", fontsize=12, fontweight="bold")

for d in range(PRED_HORIZON):
    err_map = np.mean([np.abs(r["pred"][d, 0] - r["target"][d, 0])
                       for r in all_results], axis=0)
    ax = axes[d]
    im = ax.imshow(err_map, cmap="hot", vmin=0, vmax=err_map.max())
    ax.set_title(f"Day+{d+1}  (mean MAE={err_map.mean():.4f})")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
p = os.path.join(EXT_DIR, "spatial_error_heatmap.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. IoU SCATTER  (Day+1 vs Day+2 vs Day+3)
# ═══════════════════════════════════════════════════════════════════════════════
iou_d = [[r["day_metrics"][d]["IoU"] for r in all_results] for d in range(PRED_HORIZON)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("IoU Correlation Across Prediction Days", fontsize=12, fontweight="bold")
pairs = [(0,1),(0,2),(1,2)]
for ax, (a, b) in zip(axes, pairs):
    ax.scatter(iou_d[a], iou_d[b], alpha=0.4, s=10, color="#3498db")
    lims = [min(min(iou_d[a]), min(iou_d[b])), 1.0]
    ax.plot(lims, lims, "r--", lw=1)
    corr = np.corrcoef(iou_d[a], iou_d[b])[0,1]
    ax.set_xlabel(f"IoU Day+{a+1}")
    ax.set_ylabel(f"IoU Day+{b+1}")
    ax.set_title(f"Day+{a+1} vs Day+{b+1}  (r={corr:.3f})")
    ax.grid(alpha=0.3)

plt.tight_layout()
p = os.path.join(EXT_DIR, "iou_scatter.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. THRESHOLD SWEEP  (IoU & F1 vs threshold)
# ═══════════════════════════════════════════════════════════════════════════════
thrs = np.linspace(0.1, 0.9, 33)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Metric vs Classification Threshold", fontsize=12, fontweight="bold")

for d in range(PRED_HORIZON):
    ious, f1s = [], []
    for thr in thrs:
        iou_vals, f1_vals = [], []
        for r in all_results:
            m = compute_metrics(r["pred"][d, 0], r["target"][d, 0], threshold=thr)
            iou_vals.append(m["IoU"]); f1_vals.append(m["F1"])
        ious.append(np.mean(iou_vals)); f1s.append(np.mean(f1_vals))
    axes[0].plot(thrs, ious, color=colors_day[d], lw=2, label=f"Day+{d+1}")
    axes[1].plot(thrs, f1s,  color=colors_day[d], lw=2, label=f"Day+{d+1}")

for ax, metric in zip(axes, ["IoU", "F1"]):
    ax.set_xlabel("Threshold"); ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs Threshold")
    ax.axvline(0.5, color="gray", ls="--", lw=1, label="thr=0.5")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
p = os.path.join(EXT_DIR, "threshold_sweep.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 8. GRAD-NORM HISTORY  (from training history if available)
#    + Loss gap (train vs val overfitting check)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Overfitting & Convergence Analysis", fontsize=12, fontweight="bold")

ep = list(range(1, len(hist["train_loss"]) + 1))
gap = [v - t for v, t in zip(hist["val_loss"], hist["train_loss"])]

ax = axes[0]
ax.plot(ep, hist["train_loss"], "b-o", ms=3, label="Train")
ax.plot(ep, hist["val_loss"],   "r-o", ms=3, label="Val")
ax.fill_between(ep, hist["train_loss"], hist["val_loss"],
                alpha=0.15, color="orange", label="Gap")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Train vs Val Loss")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ep, gap, "g-o", ms=3)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.fill_between(ep, gap, 0, where=[g > 0 for g in gap],
                alpha=0.3, color="red",   label="Overfitting")
ax.fill_between(ep, gap, 0, where=[g <= 0 for g in gap],
                alpha=0.3, color="green", label="Underfitting")
ax.set_xlabel("Epoch"); ax.set_ylabel("Val − Train loss")
ax.set_title("Generalisation Gap")
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
p = os.path.join(EXT_DIR, "overfitting_analysis.png")
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); print(f"Saved: {p}")

# ═══════════════════════════════════════════════════════════════════════════════
# 9. FINAL EXTENDED SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*62}")
print(f"  EXTENDED MODEL SUMMARY")
print(f"{'═'*62}")
print(f"  Architecture     : CNN-Transformer Hybrid (v4.1)")
print(f"  Total params     : {total:,}  ({total/1e6:.2f}M)")
for comp, n in component_params.items():
    print(f"    {comp:<22}: {n:>9,}  ({100*n/total:.1f}%)")
print(f"  EMBED_DIM        : {EMBED_DIM}")
print(f"  NUM_LAYERS       : {NUM_LAYERS}")
print(f"  NUM_HEADS        : {NUM_HEADS}")
print(f"  Best val loss    : {ckpt['val_loss']:.5f}  (v3: 0.22121  →  -{(0.22121-ckpt['val_loss'])/0.22121*100:.1f}%)")
print(f"  Mean test IoU    : {np.mean([r['mean_iou'] for r in all_results]):.4f}")
print(f"  Median test IoU  : {np.median([r['mean_iou'] for r in all_results]):.4f}")
print(f"  IoU std          : {np.std([r['mean_iou'] for r in all_results]):.4f}")
print(f"  IoU > 0.8        : {sum(r['mean_iou']>0.8 for r in all_results)/len(all_results)*100:.1f}% of test samples")
print(f"  IoU > 0.9        : {sum(r['mean_iou']>0.9 for r in all_results)/len(all_results)*100:.1f}% of test samples")
print(f"  Extended plots   : {EXT_DIR}")
print(f"{'═'*62}")


══════════════════════════════════════════════════════════
  PARAMETER COUNT BY COMPONENT
══════════════════════════════════════════════════════════
  Component                     Params   % total
  ──────────────────────────────────────────────────
  encoder                    1,240,320    11.21%
  transformer                2,386,944    21.58%
  temp_pool                     16,513     0.15%
  weather_fuse                  18,304     0.17%
  decoders                   7,400,403    66.90%
  ──────────────────────────────────────────────────
  TOTAL                     11,062,484   100.00%
══════════════════════════════════════════════════════════

─────────────────────────────────────────────────────────────────
  Layer                                        Params       %
─────────────────────────────────────────────────────────────────
  encoder.enc1.conv.0.block.0                   2,880   0.03%
  encoder.enc1.conv.0.block.1                      64   0.00%
  encoder.enc1.conv.1.b

## Cell 12 — Extended Statistics & Plots

Deep-dive analysis going beyond standard evaluation metrics.

### Plots Generated

**1. Parameter Distribution** — bar chart and pie chart showing how the 11M parameters are allocated across encoder, transformer, decoder, and conditioning modules.

**2. Encoder Progression** — visual diagram of spatial resolution and channel depth at each encoder stage.

**3. Precision–Recall Curves** — full PR curves with AUC for each prediction day. More informative than a single threshold metric — reveals the precision/recall trade-off across all operating points.

**4. Calibration Plot** — compares predicted fire probability to observed fire frequency across probability bins. A well-calibrated model follows the diagonal. Deviations indicate systematic over- or under-confidence.

**5. Spatial Error Heatmap** — mean absolute error averaged across all test samples at each pixel location. Reveals whether the model systematically struggles in certain spatial regions (e.g. fire edges, corners).

**6. IoU Scatter (Day+1 vs Day+2 vs Day+3)** — shows whether prediction quality degrades over the forecast horizon and whether errors are correlated across days.

**7. Threshold Sweep** — IoU and F1 vs classification threshold from 0.1 to 0.9. Identifies the optimal operating threshold (may differ from the default 0.5).

**8. Overfitting Analysis** — generalisation gap (val − train loss) over epochs. Positive gap = overfitting, negative = underfitting. Helps diagnose whether more regularisation or more training data is needed.


In [18]:
import zipfile, os

ZIP_PATH = os.path.join(SAVE_DIR, "deepfire_v4_1_full.zip")

# ── Collect all files to include ──────────────────────────────────────────────
to_zip = []

# Checkpoint
to_zip.append(MODEL_PATH)

# Original evaluation (plots + csv)
for fname in os.listdir(PLOTS_DIR):
    if fname.endswith(".png"):
        to_zip.append(os.path.join(PLOTS_DIR, fname))
for fname in os.listdir(STATS_DIR):
    to_zip.append(os.path.join(STATS_DIR, fname))

# Extended plots
for fname in os.listdir(EXT_DIR):
    if fname.endswith(".png"):
        to_zip.append(os.path.join(EXT_DIR, fname))

# ── Write zip ─────────────────────────────────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(to_zip):
        if os.path.exists(fpath):
            arcname = os.path.relpath(fpath, SAVE_DIR)
            zf.write(fpath, arcname)
            print(f"  + {arcname}")
        else:
            print(f"  ! MISSING: {fpath}")

    names = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f"\n{'='*55}")
print(f"  Zip  : {ZIP_PATH}")
print(f"  Size : {zip_mb:.1f} MB")
print(f"  Files: {len(names)}")
print(f"{'='*55}")
print(f"\nContents by folder:")
folders = {}
for n in names:
    folder = os.path.dirname(n) or "root"
    folders.setdefault(folder, []).append(os.path.basename(n))
for folder, files in sorted(folders.items()):
    print(f"  [{folder}]")
    for f in sorted(files):
        print(f"    {f}")

  + deepfire_best.pt
  + evaluation/extended/calibration.png
  + evaluation/extended/encoder_progression.png
  + evaluation/extended/iou_scatter.png
  + evaluation/extended/overfitting_analysis.png
  + evaluation/extended/parameter_distribution.png
  + evaluation/extended/precision_recall_curves.png
  + evaluation/extended/spatial_error_heatmap.png
  + evaluation/extended/threshold_sweep.png
  + evaluation/plots/attention_analysis.png
  + evaluation/plots/best_01_sample0190.png
  + evaluation/plots/best_02_sample0284.png
  + evaluation/plots/best_03_sample0372.png
  + evaluation/plots/metric_distributions.png
  + evaluation/plots/random_01_sample0233.png
  + evaluation/plots/random_02_sample0103.png
  + evaluation/plots/random_03_sample0388.png
  + evaluation/plots/training_curves.png
  + evaluation/plots/worst_01_sample0001.png
  + evaluation/plots/worst_02_sample0173.png
  + evaluation/plots/worst_03_sample0168.png
  + evaluation/stats/test_metrics.csv

  Zip  : /kaggle/working/deepf